In [ ]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
def load_data(path):
    try:
        df = pd.read_csv(path)
        print(f'Data is loaded with : {len(df)} Rows')
        return df
    except Exception as e:
        print(f'You have a problem: {e}')
        return None

path = "../../data/raw/Clean_1.csv"
df = load_data(path)

In [ ]:
df.columns

In [ ]:
# Remove salary outliers
z_scores = np.abs((df['salary_clean'] - df['salary_clean'].mean()) / df['salary_clean'].std())
df = df[z_scores < 3]

# One-hot encode categorical variables
df= pd.get_dummies(
    df,
    columns=['job_clean', 'Size', 'Sector', 'Industry', 'state'],
    drop_first=True
)

In [ ]:
df.head()

In [ ]:
cols_to_drop = [
    "index", "Job Title", "Salary Estimate", "Job Description", "Location", 
    "Company Name", "Headquarters",  "Founded", "Type of ownership",
    "Revenue", "Competitors", "description_clean", 
    "skills"
]

df_final = df.drop(columns=cols_to_drop)
    
# Fill remaining NaNs (e.g. Rating, company_age) if any
df_final.fillna(df_final.median(), inplace=True)
    
print(f"Preprocessed data shape: {df_final.shape}")

In [24]:
df_final.head()

,Rating,salary_clean,company_age,skills_count,desc_length,title_length,job_clean_Data Engineer,job_clean_Data Scientist,job_clean_Machine Learning Engineer,job_clean_Other,job_clean_Senior,Size_10000+ employees,Size_1001 to 5000 employees,Size_201 to 500 employees,Size_5001 to 10000 employees,Size_501 to 1000 employees,Size_51 to 200 employees,Size_Unknown,Sector_Aerospace & Defense,Sector_Agriculture & Forestry,Sector_Biotech & Pharmaceuticals,Sector_Business Services,"Sector_Construction, Repair & Maintenance",Sector_Consumer Services,Sector_Education,Sector_Finance,Sector_Government,Sector_Health Care,Sector_Information Technology,Sector_Insurance,Sector_Manufacturing,Sector_Media,Sector_Non-Profit,"Sector_Oil, Gas, Energy & Utilities",Sector_Real Estate,Sector_Retail,Sector_Telecommunications,Sector_Transportation & Logistics,Sector_Travel & Tourism,Sector_Unknown,Industry_Advertising & Marketing,Industry_Aerospace & Defense,Industry_Architectural & Engineering Services,Industry_Banks & Credit Unions,Industry_Biotech & Pharmaceuticals,"Industry_Cable, Internet & Telephone Providers",Industry_Chemical Manufacturing,Industry_Colleges & Universities,Industry_Computer Hardware & Software,Industry_Construction,Industry_Consulting,Industry_Consumer Electronics & Appliances Stores,Industry_Consumer Products Manufacturing,"Industry_Department, Clothing, & Shoe Stores",Industry_Electrical & Electronic Manufacturing,Industry_Energy,Industry_Enterprise Software & Network Solutions,Industry_Express Delivery Services,Industry_Farm Support Services,Industry_Federal Agencies,Industry_Financial Transaction Processing,Industry_Food & Beverage Manufacturing,Industry_Food & Beverage Stores,Industry_Health Care Services & Hospitals,"Industry_Health, Beauty, & Fitness","Industry_Hotels, Motels, & Resorts",Industry_IT Services,Industry_Industrial Manufacturing,Industry_Insurance Agencies & Brokerages,Industry_Insurance Carriers,Industry_Internet,Industry_Investment Banking & Asset Management,Industry_Lending,Industry_Logistics & Supply Chain,Industry_Miscellaneous Manufacturing,Industry_News Outlet,Industry_Oil & Gas Services,Industry_Other Retail Stores,Industry_Rail,Industry_Real Estate,Industry_Research & Development,Industry_Shipping,Industry_Social Assistance,Industry_Staffing & Outsourcing,Industry_State & Regional Agencies,Industry_Telecommunications Manufacturing,Industry_Telecommunications Services,Industry_Timber Operations,Industry_Transportation Equipment Manufacturing,Industry_Transportation Management,Industry_Travel Agencies,Industry_Unknown,Industry_Utilities,Industry_Venture Capital & Private Equity,Industry_Video Games,Industry_Wholesale,state_AZ,state_CA,state_CO,state_CT,state_DC,state_FL,state_GA,state_IA,state_IL,state_IN,state_KS,state_LA,state_MA,state_MD,state_MI,state_MN,state_MO,state_MS,state_NC,state_NE,state_NH,state_NJ,state_NY,state_OH,state_OK,state_OR,state_PA,state_RI,state_SC,state_TN,state_TX,state_UT,state_VA,state_WA,state_WI,state_WV
0,3.1,154000.0,31.0,54,3389,17,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,4.2,154000.0,56.0,63,4076,14,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,

In [ ]:
X = df_final.drop(columns=['salary_clean'])
y = df_final['salary_clean']

In [25]:
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor

# Split (keep same split for fair comparison)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 1️⃣ Random Forest
# -----------------------------
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=4,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest")
print("R2:", r2_score(y_test, rf_pred))
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("-" * 40)


# -----------------------------
# 2️⃣ Gradient Boosting
# -----------------------------
gb = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

print("Gradient Boosting")
print("R2:", r2_score(y_test, gb_pred))
print("MAE:", mean_absolute_error(y_test, gb_pred))
print("-" * 40)


# -----------------------------
# 3️⃣ XGBoost
# -----------------------------
xgbr = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgbr.fit(X_train, y_train)
xgb_pred = xgbr.predict(X_test)

print("XGBoost")
print("R2:", r2_score(y_test, xgb_pred))
print("MAE:", mean_absolute_error(y_test, xgb_pred))

Random Forest
R2: -0.011031263600127739
MAE: 21646.50969252019
----------------------------------------
Gradient Boosting
R2: -0.24642670906108322
MAE: 23776.670963727123
----------------------------------------
XGBoost
R2: -0.18030545398426168
MAE: 22553.771916746184
